<a href="https://colab.research.google.com/github/Arjunyadav0008/EDA-Employee-Promotion-23BDS0101/blob/main/EDA_Complete_Lab_Employee_Promotion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exploratory Data Analysis (EDA) Lab — Complete Practical Notebook

**Dataset:** Employee Promotion Prediction (`employee_promotion.csv`)  
**Course:** BCSE331P — Exploratory Data Analysis Lab  
**Name:** Arjun Yadav  
**Registration No.:** 23BDS0101

This notebook covers the supplied lab curriculum as comprehensively as possible using the employee-promotion dataset. It includes data loading, transformation and preprocessing, descriptive statistics, covariance/correlation, outlier detection, missing-data estimation and imputation, structured EDA, K-means clustering, and K-nearest-neighbour classification.

> **How to use:** Run the notebook from top to bottom in Google Colab or Jupyter. The dataset is downloaded from the URL below. Some analyses automatically adapt to the dataset's actual columns. Imputation and scaling are demonstrated on copies so the original data remains available for comparison. For supervised KNN, the notebook detects a likely target column (normally `is_promoted`); check the printed target before interpreting results.

## 1. Setup and load the dataset

The code first tries the raw GitHub URL. If the download is unavailable, upload `employee_promotion.csv` into the notebook session and rerun the fallback cell.

In [ ]:
# If running locally and a package is missing, uncomment:
# %pip install pandas numpy matplotlib seaborn scikit-learn scipy

import warnings
warnings.filterwarnings("ignore")

import io
import os
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, silhouette_score
)
from sklearn.cluster import KMeans
from sklearn.neighbors import KNeighborsClassifier
from sklearn.decomposition import PCA

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

DATA_URL = "https://raw.githubusercontent.com/salemprakash/EDA/main/Data/employee_promotion.csv"
DATA_FILE = "employee_promotion.csv"

try:
    df = pd.read_csv(DATA_URL)
    df.to_csv(DATA_FILE, index=False)
    print("Dataset downloaded successfully.")
except Exception as e:
    print("Could not download automatically:", repr(e))
    if os.path.exists(DATA_FILE):
        df = pd.read_csv(DATA_FILE)
        print("Loaded local", DATA_FILE)
    else:
        raise RuntimeError(
            "Download failed. Upload employee_promotion.csv to the notebook session, "
            "then run: df = pd.read_csv('/content/employee_promotion.csv')"
        )

print("Shape:", df.shape)
display(df.head())
print("\nColumn names and types:")
display(pd.DataFrame({"dtype": df.dtypes.astype(str), "non_null": df.notna().sum(), "missing": df.isna().sum()}))

## 2. Dataset overview and data quality

Inspect dimensions, variable types, duplicates, unique values, and a concise data dictionary. Confirm the meaning of columns from the dataset source before drawing business conclusions.

In [ ]:
print("Rows:", df.shape[0], "| Columns:", df.shape[1])
print("\nInfo:")
df.info()

print("\nDescriptive summary (all columns):")
display(df.describe(include="all").T)

print("\nDuplicate rows:", df.duplicated().sum())
print("\nUnique values per column:")
display(df.nunique(dropna=False).sort_values(ascending=False).rename("n_unique").to_frame())

print("\nMissing values per column:")
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}).sort_values("missing_percent", ascending=False)
display(missing_summary)

## 3. Data transformation and preprocessing

Curriculum requirement: transform data to ranges **[-3, +3]**, **[-1, +1]**, and **[0, 1]**. The notebook demonstrates z-score standardisation, min-max scaling, and a direct [-1, 1] mapping. Scaling is performed on numeric columns only; missing values are median-filled in these demonstration copies.

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
print("Numeric columns:", numeric_cols)

if numeric_cols:
    X_num = df[numeric_cols].copy()
    X_num = X_num.fillna(X_num.median(numeric_only=True))

    # [0, 1] min-max scaling
    scaler_01 = MinMaxScaler(feature_range=(0, 1))
    scaled_01 = pd.DataFrame(scaler_01.fit_transform(X_num), columns=numeric_cols, index=df.index)

    # [-1, 1] min-max scaling
    scaler_m11 = MinMaxScaler(feature_range=(-1, 1))
    scaled_m11 = pd.DataFrame(scaler_m11.fit_transform(X_num), columns=numeric_cols, index=df.index)

    # Standard z-score scaling; clipping to [-3, 3] demonstrates the requested range.
    z = (X_num - X_num.mean()) / X_num.std(ddof=0).replace(0, 1)
    scaled_m33 = z.clip(-3, 3)

    print("Scaled to [0, 1] — preview")
    display(scaled_01.head())
    print("Scaled to [-1, 1] — preview")
    display(scaled_m11.head())
    print("Z-scores clipped to [-3, 3] — preview")
    display(scaled_m33.head())

    print("Observed ranges:")
    display(pd.DataFrame({
        "min_[0,1]": scaled_01.min(),
        "max_[0,1]": scaled_01.max(),
        "min_[-1,1]": scaled_m11.min(),
        "max_[-1,1]": scaled_m11.max(),
        "min_clipped_z": scaled_m33.min(),
        "max_clipped_z": scaled_m33.max()
    }))
else:
    print("No numeric columns found to scale.")

## 4. Summary statistics: mean, median, mode, variance, standard deviation

Mode is calculated safely for numeric and categorical variables. For numeric columns, the full summary is also shown in a single table.

In [ ]:
summary_rows = []
for col in df.columns:
    s = df[col]
    non_missing = s.dropna()
    modes = non_missing.mode()
    summary_rows.append({
        "column": col,
        "dtype": str(s.dtype),
        "count": int(s.count()),
        "mean": s.mean() if pd.api.types.is_numeric_dtype(s) else np.nan,
        "median": s.median() if pd.api.types.is_numeric_dtype(s) else np.nan,
        "mode": ", ".join(map(str, modes.head(5).tolist())) if len(modes) else np.nan,
        "variance": s.var() if pd.api.types.is_numeric_dtype(s) else np.nan,
        "std_dev": s.std() if pd.api.types.is_numeric_dtype(s) else np.nan
    })
summary_stats = pd.DataFrame(summary_rows).set_index("column")
display(summary_stats)

if numeric_cols:
    display(df[numeric_cols].describe().T)

## 5. Exploratory visualisations for structured data

The plots below are selected automatically based on the dataset's columns. Histograms and boxplots show numeric distributions; count plots show the most frequent categories; the correlation heatmap highlights linear relationships.

In [ ]:
# Numeric distributions
for col in numeric_cols[:12]:
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    sns.histplot(df[col].dropna(), kde=True, ax=axes[0])
    axes[0].set_title(f"Distribution: {col}")
    sns.boxplot(x=df[col], ax=axes[1])
    axes[1].set_title(f"Boxplot: {col}")
    plt.tight_layout()
    plt.show()

# Categorical distributions (limit to avoid overwhelming output)
categorical_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
for col in categorical_cols[:10]:
    plt.figure(figsize=(9, 3.5))
    order = df[col].astype("string").fillna("<MISSING>").value_counts().head(15).index
    sns.countplot(data=df.assign(**{col: df[col].astype("string").fillna("<MISSING>")}),
                  y=col, order=order)
    plt.title(f"Top categories: {col}")
    plt.tight_layout()
    plt.show()

if len(numeric_cols) >= 2:
    plt.figure(figsize=(max(7, len(numeric_cols)*0.8), max(5, len(numeric_cols)*0.6)))
    sns.heatmap(df[numeric_cols].corr(), annot=True, cmap="coolwarm", center=0, fmt=".2f")
    plt.title("Correlation heatmap (numeric variables)")
    plt.tight_layout()
    plt.show()

## 6. Covariance and correlation for X,Y values

This section computes covariance and Pearson correlation for all numeric columns, and plots pairwise relationships for a manageable subset. Correlation measures linear association, not causation.

In [ ]:
if len(numeric_cols) >= 2:
    print("Covariance matrix:")
    display(df[numeric_cols].cov())
    print("Pearson correlation matrix:")
    display(df[numeric_cols].corr(method="pearson"))
    print("Spearman correlation matrix (rank-based):")
    display(df[numeric_cols].corr(method="spearman"))

    pair_cols = numeric_cols[:5]
    if len(pair_cols) >= 2:
        sns.pairplot(df[pair_cols].dropna().sample(min(500, len(df[pair_cols].dropna())), random_state=42)
                     if len(df[pair_cols].dropna()) else df[pair_cols].dropna())
        plt.show()
else:
    print("At least two numeric columns are required for covariance/correlation.")

## 7. Outlier detection using statistical techniques

Uses the IQR rule and z-scores for numeric columns. These methods flag potential outliers; they do not prove that a value is erroneous. Do not automatically delete flagged rows without domain justification.

In [ ]:
outlier_rows = []
for col in numeric_cols:
    s = df[col].dropna()
    if len(s) < 4:
        continue
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    iqr_mask = (df[col] < lower) | (df[col] > upper)
    sd = s.std(ddof=0)
    z_scores = (df[col] - s.mean()) / (sd if sd else 1)
    z_mask = z_scores.abs() > 3
    outlier_rows.append({
        "column": col, "IQR_lower": lower, "IQR_upper": upper,
        "IQR_outlier_count": int(iqr_mask.sum()),
        "IQR_outlier_percent": round(100 * iqr_mask.mean(), 2),
        "zscore_abs_gt_3_count": int(z_mask.sum())
    })
outlier_summary = pd.DataFrame(outlier_rows)
display(outlier_summary)

# Show flagged rows for the column with the highest IQR outlier count
if not outlier_summary.empty:
    candidate = outlier_summary.sort_values("IQR_outlier_count", ascending=False).iloc[0]["column"]
    s = df[candidate].dropna()
    q1, q3 = s.quantile([0.25, 0.75]); iqr = q3 - q1
    mask = (df[candidate] < q1 - 1.5*iqr) | (df[candidate] > q3 + 1.5*iqr)
    print("Example potential outliers in:", candidate)
    display(df.loc[mask].head(20))

## 8. Missing-data estimation and imputation

Curriculum requirement: global methods, class-based methods, and multiple imputation. The notebook compares:
- **Global methods:** mean, median, or mode replacement.
- **Class-based method:** impute within groups of a detected target/class column where possible, with a global fallback for groups that are entirely missing.
- **Multiple imputation:** `IterativeImputer` (MICE-style chained regression; results depend on modelling assumptions). A simple imputation fallback is included if the environment does not support it.

The original `df` is never overwritten.

In [ ]:
# Detect likely target/class column; adjust manually if needed.
target_candidates = ["is_promoted", "is_promote", "promoted", "promotion", "target", "label", "class"]
target_col = next((c for c in df.columns if c.lower() in target_candidates), None)
if target_col is None:
    # Try a common binary/categorical outcome name
    target_col = next((c for c in df.columns if "promot" in c.lower()), None)
print("Detected class/target column for class-based imputation:", target_col)

# Global imputation comparison
global_imputed = df.copy()
for col in df.columns:
    if global_imputed[col].isna().any():
        if pd.api.types.is_numeric_dtype(global_imputed[col]):
            global_imputed[col] = global_imputed[col].fillna(global_imputed[col].median())
        else:
            mode_vals = global_imputed[col].mode(dropna=True)
            if len(mode_vals):
                global_imputed[col] = global_imputed[col].fillna(mode_vals.iloc[0])
print("Missing values after global median/mode imputation:", int(global_imputed.isna().sum().sum()))
display(pd.DataFrame({
    "before": df.isna().sum(),
    "after_global_imputation": global_imputed.isna().sum()
}))

# Class-based imputation: use class-specific median/mode, then global fallback
class_imputed = df.copy()
if target_col is not None:
    for col in class_imputed.columns:
        if col == target_col or not class_imputed[col].isna().any():
            continue
        if pd.api.types.is_numeric_dtype(class_imputed[col]):
            class_imputed[col] = class_imputed.groupby(target_col, dropna=False)[col].transform(
                lambda x: x.fillna(x.median())
            )
            class_imputed[col] = class_imputed[col].fillna(class_imputed[col].median())
        else:
            def fill_group_mode(x):
                m = x.mode(dropna=True)
                return x.fillna(m.iloc[0] if len(m) else np.nan)
            class_imputed[col] = class_imputed.groupby(target_col, dropna=False)[col].transform(fill_group_mode)
            m = class_imputed[col].mode(dropna=True)
            if len(m):
                class_imputed[col] = class_imputed[col].fillna(m.iloc[0])
    print("Missing values after class-based imputation:", int(class_imputed.isna().sum().sum()))
else:
    print("No likely class column detected; class-based imputation skipped. Set target_col manually if applicable.")

# Multiple imputation on numeric variables
numeric_missing_cols = [c for c in numeric_cols if df[c].isna().any()]
if numeric_missing_cols:
    try:
        from sklearn.experimental import enable_iterative_imputer  # noqa: F401
        from sklearn.impute import IterativeImputer
        imp = IterativeImputer(random_state=42, max_iter=10, sample_posterior=False)
        multi_numeric = pd.DataFrame(imp.fit_transform(df[numeric_cols]), columns=numeric_cols, index=df.index)
        print("IterativeImputer completed for numeric columns.")
        display(multi_numeric[numeric_missing_cols].head())
    except Exception as e:
        print("IterativeImputer unavailable:", repr(e))
        print("Using median-imputation fallback for numeric columns.")
        multi_numeric = df[numeric_cols].fillna(df[numeric_cols].median(numeric_only=True))
else:
    print("No missing numeric values detected; multiple imputation is not needed for numeric columns.")
    multi_numeric = df[numeric_cols].copy() if numeric_cols else pd.DataFrame(index=df.index)

print("Note: IterativeImputer here is a single completed dataset, not pooled inference across multiple imputed datasets.")

## 9. K-means clustering

Curriculum requirement: read data and user-specified `k`, run K-means, and display cluster characteristics. Numeric features are median-imputed and standardised. Identifier-like columns are excluded where detectable. The target label, if found, is excluded from unsupervised clustering to avoid target leakage.

In [ ]:
# Select numeric features; remove likely IDs and the target label
cluster_cols = numeric_cols.copy()
for col in cluster_cols.copy():
    name = col.lower()
    if name in ["employee_id", "employeeid", "id", "index"] or name.endswith("_id"):
        cluster_cols.remove(col)
if target_col in cluster_cols:
    cluster_cols.remove(target_col)

if len(cluster_cols) >= 2 and len(df) >= 3:
    cluster_data = df[cluster_cols].copy()
    cluster_data = cluster_data.fillna(cluster_data.median(numeric_only=True))
    cluster_scaler = StandardScaler()
    cluster_scaled = cluster_scaler.fit_transform(cluster_data)

    k = 3  # EDIT THIS VALUE to use your chosen number of clusters
    k = max(2, min(int(k), len(df)-1))
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    cluster_labels = km.fit_predict(cluster_scaled)

    clustered_df = df.copy()
    clustered_df["cluster"] = cluster_labels
    print("Cluster sizes:")
    display(clustered_df["cluster"].value_counts().sort_index().rename("rows").to_frame())
    print("Cluster characteristics (numeric feature means):")
    display(clustered_df.groupby("cluster")[cluster_cols].mean().round(3))
    print("Cluster characteristics (numeric feature medians):")
    display(clustered_df.groupby("cluster")[cluster_cols].median().round(3))

    if len(set(cluster_labels)) > 1 and len(df) > k:
        print("Silhouette score:", round(silhouette_score(cluster_scaled, cluster_labels), 4))

    # 2D PCA visualisation
    if len(cluster_cols) >= 2:
        coords = PCA(n_components=2, random_state=42).fit_transform(cluster_scaled)
        plt.figure(figsize=(8, 5))
        sns.scatterplot(x=coords[:, 0], y=coords[:, 1], hue=cluster_labels, palette="tab10", s=45)
        plt.title("K-means clusters projected into 2D using PCA")
        plt.xlabel("PC1"); plt.ylabel("PC2")
        plt.tight_layout(); plt.show()
else:
    print("K-means needs at least two usable numeric features and three rows.")

## 10. K-nearest-neighbour (KNN) classification

Curriculum requirement: implement nearest-neighbour classification. The code uses a stratified train/test split when possible, imputes missing values, one-hot encodes categorical predictors, scales numeric predictors, and evaluates KNN. The likely promotion label is used as the target. Verify that the detected target truly represents the outcome you want to predict. IDs are excluded from predictors when recognisable.

In [ ]:
# Use detected target; change TARGET_OVERRIDE manually if required.
TARGET_OVERRIDE = None  # e.g. "is_promoted"
knn_target = TARGET_OVERRIDE or target_col

if knn_target is None:
    print("KNN skipped: no likely target column detected. Set TARGET_OVERRIDE to the correct label column.")
elif knn_target not in df.columns:
    print("KNN skipped: target column not found:", knn_target)
else:
    model_df = df.loc[df[knn_target].notna()].copy()
    y_raw = model_df[knn_target]
    X = model_df.drop(columns=[knn_target]).copy()

    # Remove obvious ID columns from features
    id_cols = [c for c in X.columns if c.lower() in ["employee_id", "employeeid", "id", "index"] or c.lower().endswith("_id")]
    X = X.drop(columns=id_cols, errors="ignore")

    # Convert target to categorical labels, then encode
    y = LabelEncoder().fit_transform(y_raw.astype(str))
    if pd.Series(y).nunique() < 2:
        print("KNN skipped: target contains fewer than two classes.")
    else:
        numeric_features = X.select_dtypes(include=np.number).columns.tolist()
        categorical_features = [c for c in X.columns if c not in numeric_features]

        numeric_transformer = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ])
        categorical_transformer = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ])
        transformers = []
        if numeric_features:
            transformers.append(("num", numeric_transformer, numeric_features))
        if categorical_features:
            transformers.append(("cat", categorical_transformer, categorical_features))

        if not transformers:
            print("KNN skipped: no predictor features remain.")
        else:
            preprocessor = ColumnTransformer(transformers=transformers)
            class_counts = pd.Series(y).value_counts()
            stratify_y = y if class_counts.min() >= 2 else None
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=0.2, random_state=42, stratify=stratify_y
            )
            n_neighbors = min(5, len(X_train))
            knn = Pipeline(steps=[
                ("preprocessor", preprocessor),
                ("classifier", KNeighborsClassifier(n_neighbors=max(1, n_neighbors)))
            ])
            knn.fit(X_train, y_train)
            y_pred = knn.predict(X_test)
            print("Target:", knn_target)
            print("Train rows:", len(X_train), "| Test rows:", len(X_test))
            print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
            print("\nClassification report:")
            print(classification_report(y_test, y_pred, zero_division=0))
            print("Confusion matrix:")
            ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
            plt.title("KNN confusion matrix")
            plt.tight_layout(); plt.show()
            print("Interpretation note: accuracy alone may be misleading for imbalanced promotion labels. Inspect precision, recall, F1, and the confusion matrix.")

## 11. Class balance and simple baseline comparison

Promotion datasets are often imbalanced. This section checks class counts and compares KNN accuracy with a majority-class baseline, helping show why accuracy should not be used alone.

In [ ]:
if target_col is not None and target_col in df.columns:
    print("Target distribution:")
    display(df[target_col].value_counts(dropna=False).rename("count").to_frame())
    display((df[target_col].value_counts(normalize=True, dropna=False) * 100).round(2).rename("percent").to_frame())
    nonmissing_target = df[target_col].dropna()
    if len(nonmissing_target):
        majority_share = nonmissing_target.value_counts(normalize=True).max()
        print("Majority-class baseline accuracy (approx.):", round(float(majority_share), 4))
else:
    print("No target column detected for class balance check.")

## 12. Export cleaned/imputed data and key tables

Exports are optional. The raw dataset remains unchanged; `global_imputed` is the median/mode-imputed copy. Files are saved to the current notebook directory.

In [ ]:
global_imputed.to_csv("employee_promotion_imputed_global.csv", index=False)
summary_stats.to_csv("eda_summary_statistics.csv")
missing_summary.to_csv("eda_missingness_summary.csv")
if not outlier_summary.empty:
    outlier_summary.to_csv("eda_outlier_summary.csv", index=False)

print("Saved:")
print("- employee_promotion_imputed_global.csv")
print("- eda_summary_statistics.csv")
print("- eda_missingness_summary.csv")
if not outlier_summary.empty:
    print("- eda_outlier_summary.csv")

## 13. Lab curriculum coverage checklist

- [x] Read data from a file / inspect data structure
- [x] Transform numeric data into **[0, 1]**, **[-1, 1]**, and clipped z-score **[-3, 3]** ranges
- [x] Compute mean, median, mode, variance, and standard deviation
- [x] Compute covariance and correlation for numeric variables
- [x] Detect potential outliers using IQR and z-scores
- [x] Estimate/impute missing data using global and class-based methods
- [x] Demonstrate iterative (MICE-style) imputation for numeric missing values when needed
- [x] Perform structured EDA with plots and summary tables
- [x] Run K-means clustering with editable `k` and inspect cluster characteristics
- [x] Implement KNN classification and evaluate predictions

### Important notes for the lab report
1. Include the actual output tables and plots generated after running the notebook.
2. Explain what each chart/table shows using the observed results rather than generic claims.
3. Check the target column and class imbalance before interpreting KNN results.
4. Imputation and scaling choices should be justified; never claim that a statistical method proves causality.
5. The curriculum asks for R programs, while this deliverable is an **IPython/Jupyter notebook in Python** as requested. If your instructor requires R specifically, the same experiments should be reproduced in an R notebook.